# AI008 Training Pipeline Notebook

This notebook mirrors the Week 6 scaffold described in `training_pipeline/README.md`.

It demonstrates the pieces that already exist in this folder today:
- runtime directory setup
- config loading through `src/utils/config_loader.py`
- seed initialisation

The remaining pipeline stages are included as dummy cells so the notebook can act as a guided walkthrough while those modules are still placeholders.

## 1. Notebook Setup

The README shows this scaffold is intended to run from the `training_pipeline` directory.
For notebook use, we resolve the folder dynamically and add `src/` to `sys.path` so imports work whether the notebook is launched from the repo root or from inside `training_pipeline`.

In [19]:
from pathlib import Path
import sys


def find_training_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        direct = candidate / 'ai-ml' / 'training_pipeline'
        if direct.exists():
            return direct
        if (candidate / 'src').exists() and (candidate / 'configs').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate ai-ml/training_pipeline from the current working directory.')


TRAINING_ROOT = find_training_root(Path.cwd())
SRC_DIR = TRAINING_ROOT / 'src'

if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Training root:', TRAINING_ROOT.name)
print('Source dir    :', SRC_DIR.name)

Training root: training_pipeline
Source dir    : src


## 2. Existing Week 6 Scaffold Pieces

These imports stay inside the `training_pipeline` folder and reflect what is currently implemented.

In [20]:
from src.utils.paths import ensure_runtime_dirs, CONFIGS_DIR, LOGS_DIR, CHECKPOINTS_DIR
from src.utils.config_loader import load_config
from src.utils.seeds import set_seed

print('Imports loaded successfully.')

Imports loaded successfully.


## 3. Runtime Folder Setup

This mirrors the current `src/main.py` behaviour: create the scaffold runtime directories before the rest of the pipeline runs.

In [21]:
ensure_runtime_dirs()

runtime_dirs = {
    'configs': CONFIGS_DIR,
    'logs': LOGS_DIR,
    'checkpoints': CHECKPOINTS_DIR,
}

for name, path in runtime_dirs.items():
    print(f'{name:12} -> {path.name} | exists={path.exists()}')

configs      -> configs | exists=True
logs         -> logs | exists=True
checkpoints  -> checkpoints | exists=True


## 4. Load the Default Config

The scaffold already includes `configs/default_config.yaml`, and this notebook loads it through `src/utils/config_loader.py` (matching the current loader implementation).

In [22]:
config_path = TRAINING_ROOT / 'configs' / 'default_config.yaml'
config = load_config(config_path)

print('Config file:', config_path.name)
print('Model type:', config['model']['type'])
print('Epochs    :', config['training']['epochs'])
print('Batch size:', config['training']['batch_size'])
print('Sections  :', list(config.keys()))

Config file: default_config.yaml
Model type: random_forest
Epochs    : 50
Batch size: 32
Sections  : ['dataset', 'preprocessing', 'model', 'training', 'output']


## 5. Seed Utility Demo

Week 6 includes a simple random seed utility. Right now it only seeds Python's `random` module, but it gives us a small, testable building block in the scaffold.

In [23]:
import random

set_seed(42)
first_run = [random.randint(0, 100) for _ in range(5)]

set_seed(42)
second_run = [random.randint(0, 100) for _ in range(5)]

print('First run :', first_run)
print('Second run:', second_run)
print('Deterministic:', first_run == second_run)

First run : [81, 14, 3, 94, 35]
Second run: [81, 14, 3, 94, 35]
Deterministic: True


## 6. Pipeline Entry Point Summary

At the moment, the scaffold entrypoint effectively does three things:
1. creates runtime folders
2. loads the selected config via `utils.config_loader`
3. prints a couple of key config values

The next cell reproduces that flow in notebook form.

In [24]:
ensure_runtime_dirs()
config = load_config(TRAINING_ROOT / 'configs' / 'default_config.yaml')

print('AI008 training pipeline scaffold is ready.')
print('Model :', config['model']['type'])
print('Epochs:', config['training']['epochs'])

AI008 training pipeline scaffold is ready.
Model : random_forest
Epochs: 50


## 7. Dummy Cells for Later Week 6 Functionality

The README lists the remaining modules as part of the scaffold, but most of them are still placeholders that raise `NotImplementedError`. These cells make that explicit and give future contributors a clear place to continue.

In [25]:
# W6-T3 placeholder: dataset loading
from src.data.dataset_loader import load_dataset

try:
    load_dataset('data/raw/example.csv')
except NotImplementedError as exc:
    print('Dataset loader placeholder:', exc)

Dataset loader placeholder: W6-T3 will implement dataset loading here.


In [26]:
# W6-T3 placeholder: dataset splitting
from src.data.splitter import split_dataset

try:
    split_dataset(data='mock-dataset', random_seed=42)
except NotImplementedError as exc:
    print('Dataset splitter placeholder:', exc)

Dataset splitter placeholder: W6-T3 will implement split logic here.


In [27]:
# W6-T4 placeholder: preprocessing and feature loading
from src.preprocessing.preprocess import preprocess_features
from src.preprocessing.feature_loader import load_feature_set

for label, fn, arg in [
    ('Feature loader', load_feature_set, 'ai004-output-placeholder'),
    ('Preprocessing', preprocess_features, 'mock-features'),
]:
    try:
        fn(arg)
    except NotImplementedError as exc:
        print(f'{label} placeholder:', exc)

Feature loader placeholder: W6-T4 will implement feature loading here.
Preprocessing placeholder: W6-T4 will implement preprocessing logic here.


In [ ]:
# W6-T5 config-based functionality check with Iris dataset
try:
    from sklearn.datasets import load_iris
    from sklearn.model_selection import train_test_split
    from src.core.trainer import TrainingConfig, GenericTrainingEngine
    from src.utils.config_loader import load_config

    def resolve_training_config(pipeline_config):
        if hasattr(TrainingConfig, 'from_pipeline_config'):
            return TrainingConfig.from_pipeline_config(pipeline_config, verbose=False)

        model_type_raw = str(pipeline_config.get('model', {}).get('type', '')).strip().lower()
        mapping = {
            'random_forest': ('sklearn', 'random_forest'),
            'isolation_forest': ('sklearn', 'isolation_forest'),
            'pytorch_mlp': ('pytorch', 'simple_mlp'),
        }
        if model_type_raw not in mapping:
            raise ValueError(f"Unsupported model.type in config: {model_type_raw!r}")

        engine_type, model_name = mapping[model_type_raw]
        training_cfg = pipeline_config.get('training', {})
        model_params = dict(pipeline_config.get('model', {}).get('hyperparameters', {}))
        return TrainingConfig(
            model_type=engine_type,
            model_name=model_name,
            model_params=model_params,
            epochs=int(training_cfg.get('epochs', 10)),
            batch_size=int(training_cfg.get('batch_size', 32)),
            learning_rate=float(training_cfg.get('learning_rate', 0.001)),
            verbose=False,
        )

    pipeline_config = load_config(TRAINING_ROOT / 'configs' / 'default_config.yaml')
    trainer_config = resolve_training_config(pipeline_config)

    iris = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data, iris.target, test_size=0.2, random_state=42 # type: ignore
    )

    if trainer_config.model_type != 'sklearn':
        print('Config model is not sklearn, skipping Iris classifier check:', pipeline_config['model']['type'])
    else:
        engine = GenericTrainingEngine(trainer_config)
        train_result = engine.fit(X_train, y_train)
        preds = engine.predict(X_test)
        accuracy = float((preds == y_test).mean())

        print('Config model     :', pipeline_config['model']['type'])
        print('Resolved engine  :', trainer_config.model_type, '/', trainer_config.model_name)
        print('Hyperparameters  :', trainer_config.model_params)
        print('Train result     :', train_result)
        print('Predictions      :', len(preds))
        print('Iris accuracy    :', round(accuracy, 4))

        assert train_result['status'] == 'trained'
        assert len(preds) == len(y_test)
except ImportError as exc:
    print('W6-T5 config-based Iris test skipped (dependency missing):', exc)


Config model     : random_forest
Resolved engine  : sklearn / random_forest
Hyperparameters  : {'n_estimators': 100, 'max_depth': 10}
Train result     : {'model_type': 'sklearn', 'status': 'trained'}
Predictions      : 30
Iris accuracy    : 1.0


In [29]:
# W6-T6 placeholder: evaluation
from src.evaluation.metrics import compute_metrics
from src.evaluation.validator import validate_predictions

for label, fn, kwargs in [
    ('Metrics', compute_metrics, {'y_true': [0, 1], 'y_pred': [0, 1]}),
    ('Validator', validate_predictions, {'y_true': [0, 1], 'y_pred': [0, 1]}),
]:
    try:
        fn(**kwargs)
    except NotImplementedError as exc:
        print(f'{label} placeholder:', exc)

Metrics placeholder: W6-T6 will implement metric calculations here.
Validator placeholder: W6-T6 will implement validation flow here.


In [30]:
# W6-T7 and W6-T8 placeholder note
print('Checkpoint management and logging are listed in the scaffold README,')
print('but this notebook leaves them as future extension points until those modules are implemented.')

Checkpoint management and logging are listed in the scaffold README,
but this notebook leaves them as future extension points until those modules are implemented.


## Next Step

As the Week 6 modules are implemented, the dummy cells above can be replaced with real examples while keeping this notebook as the main guided walkthrough for the `training_pipeline` folder.